### Wer spielt fair? – Lösung Schritt für Schritt

*30. Bundeswettbewerb Informatik (2011), 1. Runde, Aufgabe 1*

Bob spielt mit seinen Freunden oft „Schere, Stein, Papier“, wenn es darum geht, eine Entscheidung zu fällen. In einer Spielrunde wählen die beiden Spieler gleichzeitig entweder „Schere“, „Stein“ oder „Papier“. Sind die gewählten Begriffe unterschiedlich, ergibt sich aus einem Bild, wer den Zug für sich entscheiden konnte (Schere schneidet Papier, Stein schleift Schere, Papier wickelt Stein ein).

Nun hat Bob ein Programm geschrieben, das „Schere, Stein, Papier“ gegen einen Menschen spielen kann. Er behauptet, sein Programm spiele besser als ein Mensch. Seine Freunde sagen, sein Programm schummelt, denn bei Bobs Programm muss zuerst der Mensch seine Wahl eingeben, bevor der Computer seine Wahl ausgibt.

Bob sucht also nach einem Verfahren, mit dem das Programm nachweisen kann, dass es fair spielt (also seine Wahl festgelegt hat, bevor der Mensch seine Wahl eingibt), aber natürlich ohne seine Wahl vorher zu verraten.

Gleichzeitig soll das Verfahren praktikabel sein: Der Mensch soll im Nachhinein möglichst einfach überprüfen können, ob das Programm fair gespielt hat, aber während des zügig durchgeführten Spiels nicht aus möglichen Ausgaben des Programms dessen Wahl erraten können.

#### Aufgabe 1

Schlage Bob ein geeignetes Verfahren vor. Begründe, warum dein Vorschlag alle oben genannten Anforderungen erfüllt. Wenn du magst, kannst du das Programm und dein Verfahren gerne implementieren, gefordert ist das aber nicht.

**Zu den Daten:** Beispieldaten gibt es zu dieser Aufgabe nicht. Es ist eine Entwurfsaufgabe: Gefragt ist ein Verfahren mit Begründung. Wir entwerfen es, probieren es aus und greifen es dann selbst an.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Festlegen, ohne zu verraten (Commitment):** Das Programm legt seinen Zug *vorher* fest und zeigt etwas, aus dem man den Zug nicht ablesen kann. Es soll aber später nichts anderes mehr behaupten können. In der Kryptographie heißt das *Commitment*. Man kann es sich wie einen verschlossenen Umschlag vorstellen, den man vor dem Zug hinlegt und erst danach öffnet.
- **Hash-Funktion:** Sie macht aus einem beliebigen Text eine Zeichenkette fester Länge (den *Hashwert*). Derselbe Text ergibt immer denselben Hash. Aus dem Hash den Text zurückzurechnen ist praktisch unmöglich, und einen anderen Text mit demselben Hash findet man nicht.
- **Zufall als Salz:** Gibt es nur wenige Möglichkeiten (hier drei Züge), kann man alle durchprobieren. Ein zufälliger Zusatz im Text macht das Probieren aussichtslos.
- **Angreifer-Denken:** Ein Verfahren prüft man, indem man versucht, es zu überlisten, und zwar von beiden Seiten (Programm und Spieler).

**Nützliche Funktionen und Bibliotheken**
- `hashlib.sha256(text.encode()).hexdigest()`: berechnet den SHA-256-Hash eines Textes als Folge von 64 Hexadezimalziffern.
- `secrets.token_hex(8)`: erzeugt 16 zufällige Hexadezimalziffern, geeignet für Sicherheitszwecke (besser als `random`, dessen Zahlen man vorhersagen kann).
- `secrets.choice(liste)`: wählt zufällig ein Element, ebenfalls kryptographisch geeignet.
- `str.split()`, f-Strings, `in` für Listen.

## Die Idee in Worten

**Was wir erreichen wollen.** Vor dem Zug des Menschen zeigt das Programm eine Zeichenkette, die es auf seinen Zug festlegt (nachträglich kann es sie nicht mehr ändern) und die trotzdem nichts über den Zug verrät. Nach dem Zug des Menschen deckt das Programm auf. Der Mensch rechnet nach.

**Das Verfahren (mit Hash-Funktion):**
1. Das Programm wählt seinen Zug (Schere, Stein oder Papier).
2. Es baut einen Text aus **zufälligen Zeichen, dem Zug und einem Zeitstempel**, zum Beispiel `jsielshfpe Stein 2011-11-11-11:11:11`.
3. Es berechnet davon den **SHA-256-Hash** und zeigt dem Menschen *nur den Hash*.
4. Der Mensch gibt seinen Zug ein.
5. Das Programm zeigt den **ganzen Text**. Daraus liest man den Zug des Programms ab und sieht, wer gewonnen hat.
6. Der Mensch berechnet den Hash des gezeigten Textes selbst (mit einem beliebigen SHA-256-Werkzeug) und vergleicht ihn mit dem Hash aus Schritt 3.

**Warum das funktioniert** (ausführlich weiter unten, hier die Kurzfassung):
- *Das Programm kann nicht schummeln:* Es müsste nach dem Zug des Menschen einen anderen Text mit demselben Hash vorzeigen. Das ist für SHA-256 nach heutigem Wissen nicht möglich (es sind keine Kollisionen bekannt).
- *Der Mensch kann nicht mogeln:* Aus dem Hash lässt sich der Zug nicht ablesen, und durch Ausprobieren der drei Züge kommt man nicht weiter, weil die zufälligen Zeichen den Text unbekannt machen.
- *Praktikabel:* Hash anzeigen, danach Text anzeigen. Zum Nachrechnen genügt ein Online-Werkzeug oder ein Programm mit einer Zeile Code. Es braucht keine dritte Person.

Wir bauen es jetzt in Python und greifen es dann an.

## Schritt 1: Was ein Hash ist

Wir sehen uns die Hash-Funktion an einem Beispiel an. Es stammt aus den Lösungshinweisen, deshalb kennen wir das richtige Ergebnis: Für die Zeichenkette `jsielshfpe Stein jskae 2011-11-11-11:11:11` steht dort der Hash `306ad85f…909e4f`.

In [ ]:
import hashlib

def hash_von(text):
    """SHA-256 eines Textes als 64 Hexadezimalziffern."""
    return hashlib.sha256(text.encode()).hexdigest()

beispiel = "jsielshfpe Stein jskae 2011-11-11-11:11:11"
print(hash_von(beispiel))
print(len(hash_von(beispiel)), "Zeichen")
print(hash_von(beispiel) == "306ad85fa307e226a00930a0432820a98dc57938a2fd91f4fc49c62288909e4f")

306ad85fa307e226a00930a0432820a98dc57938a2fd91f4fc49c62288909e4f
64 Zeichen
True


Das stimmt mit den Hinweisen überein. Zwei Eigenschaften sehen wir gleich selbst: Der Hash hat immer dieselbe Länge, auch bei sehr kurzem oder sehr langem Text. Und eine winzige Änderung des Textes ändert den Hash komplett (*Lawineneffekt*). Man sieht dem Hash also nicht an, dass zwei Texte fast gleich sind.

In [ ]:
for text in ["Stein", "stein", "Stein ", "Stein" * 1000]:
    print(f"{len(text):5d} Zeichen -> {hash_von(text)[:20]}...")

    5 Zeichen -> 795b5603753a3fd54d66...
    5 Zeichen -> 470b8720f9f865ed7e6f...
    6 Zeichen -> e174591f2aa348c0b45e...
 5000 Zeichen -> 9a7d82b2f75754772000...


## Schritt 2: Der naive Versuch (nur den Zug hashen)

Der einfachste Gedanke: Das Programm zeigt `hash("Stein")` und deckt später „Stein“ auf. Festlegen klappt, denn das Programm kann später nichts anderes mehr aufdecken. Aber es wird auch alles **verraten**: Es gibt nur drei mögliche Züge. Der Mensch berechnet die drei Hashwerte selbst und vergleicht.

In [ ]:
ZUEGE = ["Schere", "Stein", "Papier"]

gezeigter_hash = hash_von("Stein")          # das zeigt das Programm vor dem Zug

# Der Mensch probiert einfach alle drei Zuege durch:
for zug in ZUEGE:
    if hash_von(zug) == gezeigter_hash:
        print("Der Mensch weiß jetzt vorher: Das Programm spielt", zug)

Der Mensch weiß jetzt vorher: Das Programm spielt Stein


Das Verfahren ist also **unbrauchbar**, weil der Mensch nach drei Versuchen den Zug kennt und dann immer gewinnen kann. Der Fehler ist allgemein: Wenn es nur *wenige* mögliche Eingaben gibt, hilft keine Hash-Funktion, denn man probiert alle durch. Wir brauchen mehr Möglichkeiten, die der Mensch nicht kennt: **zufällige Zeichen**.

## Schritt 3: Mit zufälligen Zeichen und Zeitstempel

Der Text besteht jetzt aus `zufall zug zeit`. Die Funktion `festlege` baut den Text und gibt Text und Hash zurück. Der Text bleibt geheim, der Hash wird gezeigt. Damit die Ausgaben im Notebook reproduzierbar sind, übergeben wir das Zufallsstück hier als Parameter. Das echte Programm nimmt `secrets.token_hex(8)` (siehe Schritt 5).

In [ ]:
def festlege(zug, zufall, zeit):
    """Baut den geheimen Text und den Hash, der vor dem Zug des Menschen gezeigt wird."""
    text = f"{zufall} {zug} {zeit}"
    return text, hash_von(text)

text, h = festlege("Stein", "jsielshfpe", "2011-11-11-11:11:11")
print("Text (geheim):", text)
print("Hash (öffentlich):", h[:30] + "...")

Text (geheim): jsielshfpe Stein 2011-11-11-11:11:11
Hash (öffentlich): 8fb53e343b190d382b54479bb64567...


Nun der Spieler: Er bekommt nach dem Zug den Text und prüft zweierlei. Erstens: Passt der Hash zum Text? Zweitens: Welchen Zug nennt der Text? Der Zug steht in der Mitte (das zweite „Wort“ des Textes). Ein Text mit anderem Aufbau wird abgelehnt.

In [ ]:
def pruefe(gezeigter_hash, aufgedeckter_text):
    """Gibt den Zug aus dem Text zurueck, wenn der Hash passt, sonst None."""
    if hash_von(aufgedeckter_text) != gezeigter_hash:
        return None                         # Text passt nicht zur Festlegung
    teile = aufgedeckter_text.split()
    if len(teile) != 3 or teile[1] not in ZUEGE:
        return None                         # richtiger Hash, aber kein gueltiger Aufbau
    return teile[1]

print(pruefe(h, text))                                      # ehrliches Programm
print(pruefe(h, "jsielshfpe Papier 2011-11-11-11:11:11"))   # Programm will nachtraeglich Papier zeigen
print(pruefe(h, "jsielshfpe stein 2011-11-11-11:11:11"))    # schon ein kleiner Buchstabe faellt auf

Stein
None
None


Wer den Text ändert (auch nur um einen Buchstaben), bekommt einen anderen Hash, und die Prüfung meldet `None`. Das ist die Fairness gegenüber dem Menschen.

## Schritt 4: Kann das Programm schummeln?

Das Programm möchte schummeln: Es hat den Hash für „Stein“ gezeigt, sieht dann, dass der Mensch „Papier“ gewählt hat, und möchte jetzt „Schere“ vorweisen. Es braucht dazu einen Text mit dem Zug „Schere“, der denselben Hash hat. Es kann dazu beliebige Zufallsstücke probieren, aber SHA-256 hat $2^{256}$ mögliche Hashwerte. Wir lassen es nur versuchen, wenigstens die **ersten 4 Hexziffern** des Hashs zu treffen (das sind $16^4 = 65\,536$ Möglichkeiten) und zählen die Versuche. Ein Treffer in 4 Ziffern ist noch lange keine Kollision, das zeigt die zweite Ausgabezeile.

In [ ]:
ziel = h[:4]                  # nur die ersten 4 Hexziffern muessen passen (ein winziger Teil des Hashs)
versuche_s = 0
gefunden = None
while gefunden is None:
    versuche_s += 1
    kandidat = f"{versuche_s} Schere 2011-11-11-11:11:11"
    if hash_von(kandidat).startswith(ziel):
        gefunden = kandidat

print("Versuche:", versuche_s, "| gefunden:", gefunden)
print("Erste 4 Ziffern gleich:", hash_von(gefunden)[:4] == ziel)
print("Ganzer Hash gleich:", hash_von(gefunden) == h)

Versuche: 18489 | gefunden: 18489 Schere 2011-11-11-11:11:11
Erste 4 Ziffern gleich: True
Ganzer Hash gleich: False


Schon für **4 von 64** Hexziffern braucht das Programm im Schnitt 65 536 Versuche. Jede weitere Ziffer macht es 16-mal so viel, für alle 64 Ziffern sind es im Schnitt $16^{64}$ Versuche, eine Zahl mit 77 Stellen. Das ist in einem zügigen Spiel nicht zu schaffen (und auch sonst nicht in der Lebenszeit des Universums). Ein Text mit dem gleichen Hash wäre eine *Kollision* (zwei verschiedene Texte, gleicher Hash). Für SHA-256 ist bisher keine bekannt, so schreiben es auch die Lösungshinweise.

Fazit: **Das Programm kann sich nach dem Zug des Menschen nicht mehr umentscheiden.**

## Schritt 5: Das ganze Spiel

Jetzt bauen wir eine Runde zusammen, so wie Bobs Programm sie ablaufen lassen würde. Das Zufallsstück erzeugt `secrets.token_hex(8)` (16 Hexziffern, also $16^{16} = 2^{64} \approx 1{,}8 \cdot 10^{19}$ Möglichkeiten). Den Zeitstempel liefert `datetime`. Die Gewinnregel steht in einem kleinen Wörterbuch.

In [ ]:
import secrets
from datetime import datetime

SCHLAEGT = {"Schere": "Papier", "Stein": "Schere", "Papier": "Stein"}   # Schere schlaegt Papier usw.

def gewinner(mensch, computer):
    if mensch == computer:
        return "unentschieden"
    return "Mensch" if SCHLAEGT[mensch] == computer else "Computer"

def spiele_runde(mensch_zug):
    """Eine Runde mit Festlegung. Gibt (gezeigter Hash, aufgedeckter Text, Zug laut Pruefung, Sieger) zurueck."""
    # 1. Programm legt sich fest
    zug = secrets.choice(ZUEGE)
    zufall = secrets.token_hex(8)
    zeit = datetime.now().strftime("%Y-%m-%d-%H:%M:%S")
    text, gezeigter = festlege(zug, zufall, zeit)
    # 2. Programm zeigt nur den Hash; der Mensch zieht (hier als Parameter)
    # 3. Programm deckt den Text auf; der Mensch prueft
    geprueft = pruefe(gezeigter, text)
    return gezeigter, text, geprueft, gewinner(mensch_zug, geprueft)

# 3000 Runden: Die Pruefung muss jedes Mal bestehen; die Zuege des Programms sollten gleichmaessig streuen.
anzahl = {z: 0 for z in ZUEGE}
alle_ok = True
for i in range(3000):
    gezeigter, text, geprueft, sieger = spiele_runde(ZUEGE[i % 3])
    alle_ok = alle_ok and (geprueft is not None) and hash_von(text) == gezeigter
    anzahl[geprueft] += 1

print("Alle Prüfungen bestanden:", alle_ok)
print("Jeder Zug des Programms zwischen 800 und 1200 Mal:", all(800 < n < 1200 for n in anzahl.values()))

Alle Prüfungen bestanden: True
Jeder Zug des Programms zwischen 800 und 1200 Mal: True


Die Ausgabe zeigt nur Wahrheitswerte, weil die Zufallszahlen bei jedem Lauf anders sind. Eine einzelne Runde sieht zum Beispiel so aus (die Zahlen sind erfunden und nur zur Anschauung):

```
Programm zeigt:     Hash 9f2c41a07b...
Mensch zieht:       Papier
Programm deckt auf: 3be90a1c5d44f7a2 Stein 2026-11-14-10:03:12
Mensch rechnet:     Hash des Textes = 9f2c41a07b...  -> gleich, Programm hat fair gespielt
Ergebnis:           Papier wickelt Stein ein, Mensch gewinnt
```

## Schritt 6: Warum die zufälligen Zeichen nötig sind (Angriff des Menschen)

Der Mensch könnte schummeln wollen: Er sieht den Hash und möchte vor seinem Zug herausfinden, was im Text steckt. Wir lassen den Spieler dazu gegen ein Verfahren **ohne** Zufallsstück antreten, also nur `zug zeit`. Der Zeitstempel ist auf die Sekunde genau. Der Mensch weiß ungefähr, wann das Programm den Hash berechnet hat (zum Beispiel in der letzten Minute), und probiert alle Sekunden und alle Züge durch.

In [ ]:
def ohne_zufall(zug, zeit):
    return hash_von(f"{zug} {zeit}")

gezeigter_oz = ohne_zufall("Papier", "2011-11-14-21:56:23")   # das Programm hat um 21:56:23 gerechnet

versuche_m = 0
treffer = None
for sekunde in range(60):                                     # die ganze Minute 21:56
    for zug in ZUEGE:
        versuche_m += 1
        if ohne_zufall(zug, f"2011-11-14-21:56:{sekunde:02d}") == gezeigter_oz:
            treffer = (zug, sekunde)

print("Versuche:", versuche_m, "-> Treffer:", treffer)

Versuche: 180 -> Treffer: ('Papier', 23)


Nach höchstens 180 Versuchen kennt der Mensch den Zug, und das Verfahren wäre wertlos. Mit den **zufälligen Zeichen** muss er auch sie raten:

In [ ]:
mit_zufall = 16 ** 16                              # secrets.token_hex(8): 16 Hexziffern
print("Möglichkeiten für das Zufallsstück:", mit_zufall)
print("Dazu 60 Sekunden und 3 Züge:", mit_zufall * 60 * 3)
print("Gegenüber ohne Zufall:", 60 * 3, "Versuche")

Möglichkeiten für das Zufallsstück: 18446744073709551616
Dazu 60 Sekunden und 3 Züge: 3320413933267719290880
Gegenüber ohne Zufall: 180 Versuche


Mehr als $3 \cdot 10^{21}$ Versuche sind in der Zeit eines Spielzugs (wenige Sekunden) nicht zu schaffen. Das Verfahren verrät dem Menschen vor dem Zug also nichts, auch wenn er genau weiß, wie der Text aufgebaut ist. Die Sicherheit liegt im Zufall, nicht in einem geheimen Verfahren. Das ist ein Grundsatz der Kryptographie: Das Verfahren darf bekannt sein.

Das Zufallsstück wird bei **jeder Runde neu** gezogen. Und es muss echter Zufall sein (`secrets` statt `random`), weil man bei `random` aus mehreren beobachteten Zufallszahlen die nächsten vorhersagen könnte.

## Schritt 7: Weitere Fälle, die das Verfahren aushalten muss

In [ ]:
h2 = festlege("Papier", "a1b2c3d4e5f60718", "2011-11-14-21:56:23")[1]
pruefungen = {
    "ehrlich aufgedeckt":          pruefe(h2, "a1b2c3d4e5f60718 Papier 2011-11-14-21:56:23"),
    "anderer Zug eingesetzt":      pruefe(h2, "a1b2c3d4e5f60718 Stein 2011-11-14-21:56:23"),
    "anderes Zufallsstück":        pruefe(h2, "0000000000000000 Papier 2011-11-14-21:56:23"),
    "anderer Zeitstempel":         pruefe(h2, "a1b2c3d4e5f60718 Papier 2011-11-14-21:56:24"),
    "gar kein Text":               pruefe(h2, ""),
    "Hash passt, Aufbau ungültig": pruefe(hash_von("Papier Stein"), "Papier Stein"),
}
for name, erg in pruefungen.items():
    print(f"{name:30s} -> {erg}")

ehrlich aufgedeckt             -> Papier
anderer Zug eingesetzt         -> None
anderes Zufallsstück           -> None
anderer Zeitstempel            -> None
gar kein Text                  -> None
Hash passt, Aufbau ungültig    -> None


Nur der ehrlich aufgedeckte Text besteht die Prüfung. (Der letzte Fall zeigt, warum `pruefe` auch den Aufbau kontrolliert: Selbst wenn ein Text zum Hash passt, muss er die Form `zufall zug zeit` haben.)

## Schritt 8: Vergleich mit anderen Verfahren

Die Lösungshinweise nennen mehrere Verfahren. Wir ordnen sie nach den Forderungen der Aufgabe (Fairness nachweisbar, Praktikabilität):

| Verfahren | Programm kann nicht schummeln? | Mensch kann nicht mogeln? | Praktikabel? |
| --- | --- | --- | --- |
| **Vertrauenswürdiger Dritter** (Zug vorher per E-Mail an eine dritte Person) | ja, wenn die Person ehrlich ist | ja | schlecht: zusätzliche Kommunikation, die Person kann mit einer Seite unter einer Decke stecken |
| **Große Textdatei** (vorab erzeugt, Programm nennt eine Zeilennummer) | ja | nur durch sehr kurze Eingabezeit | mittel: Zeitdruck für den Menschen |
| **QR-Code** (Zug und Zeit kodiert, Spieler hat 1 bis 2 Sekunden) | ja | nur durch kurze Zeit, der Code ist lesbar | gut (Handykamera), aber Zeitdruck |
| **Hash** (unser Verfahren) | ja, solange SHA-256 keine Kollisionen hat | ja, wegen der zufälligen Zeichen | gut: keine Zeitgrenze, nur ein Hash-Werkzeug nötig |
| **Asymmetrische Kryptographie** (verschlüsselt und signiert, Schlüssel erst danach) | ja | ja | schlecht: Schlüssel müssen ausgelesen und geprüft werden |

Unser Verfahren braucht **keine Zeitgrenze** und **keine dritte Person**. Die Sicherheit liegt allein in der Hash-Funktion und im Zufallsstück. Der Mensch darf beliebig lange nachdenken (die Hinweise nennen das als Vorteil gegenüber dem QR-Code).

## Grenzen des Verfahrens (ehrlich gesagt)

- **Die Sicherheit hängt von SHA-256 ab.** Sollte jemand Kollisionen finden, könnte das Programm schummeln. Nach dem Stand der Hinweise (2011) ist das nicht gelungen.
- **Der Mensch muss den Hash selbst nachrechnen.** Das ist nur ein Beweis für den, der seinem Rechenwerkzeug traut. Man kann ein Online-Werkzeug, die Kommandozeile (`sha256sum`) oder ein eigenes Programm mit `hashlib` nehmen. Das ist einfach, aber nicht „ein Blick“.
- **Mehrere Runden:** Jede Runde bekommt ein neues Zufallsstück und einen neuen Hash. Das Programm sollte nach jeder Runde den Text aufdecken, nicht nur bei Verdacht, sonst fehlt der Nachweis.

## Zum Schluss: Warum kein MiniZinc?

MiniZinc löst Probleme, in denen unter Bedingungen eine zulässige oder beste Belegung von Variablen gesucht wird (Zuordnungen, Summen, Permutationen). Bei „Wer spielt fair?“ gibt es **keine Beispieldaten** und auch **nichts zu suchen oder zu optimieren**: Gefragt ist ein Protokoll mit Begründung. Eine Hash-Funktion ist außerdem gerade so gebaut, dass man sie nicht „rückwärts“ lösen kann, das ist das Gegenteil dessen, was ein Constraint-Solver gut kann. Wir haben deshalb auf MiniZinc verzichtet.

## Für Neugierige

- **Verschlüsseln statt hashen:** Das geht auch (Hinweise: GnuPG), braucht aber mehr Technik.
- **Salz.** Das Zufallsstück heißt in der Praxis *Salt* oder *Nonce*. Dieselbe Idee schützt Passwörter in Datenbanken.
- **Commitment in der Praxis.** „Erst festlegen, dann aufdecken“ steckt in vielen Protokollen, etwa bei Online-Auktionen, bei fairen Münzwürfen am Telefon und in Blockchains.
- **Geburtstagsparadox.** Eine Kollision zu *irgendeinem* Paar von Texten zu finden ist leichter (etwa $2^{128}$ Versuche bei SHA-256) als eine zu einem vorgegebenen Hash (etwa $2^{256}$). Das Programm könnte sich also vorab zwei Texte „Stein“ und „Schere“ mit gleichem Hash suchen und später den passenden zeigen. Auch das kostet bei SHA-256 etwa $2^{128}$ Versuche und ist außer Reichweite. Bei einer kurzen Hash-Funktion (zum Beispiel mit 32 Bit) wäre es leicht, ein Grund für die Wahl von SHA-256.